In [ ]:
import pandas as pd

In [ ]:
current_resuts = 'output/LUVit_ct_by_tod_generator-2026-03-20_90-90-90.xlsx'
r_scripts_results = 'C:/Users/JKolberg/PythonProjects/control_totals/examples/legacy_luvit/output/LUVit_ct_by_tod_generator-2026-03-19_90-90-90.xlsx'

In [ ]:
df_current = pd.read_excel(current_resuts, sheet_name='unrolled')
df_original = pd.read_excel(r_scripts_results, sheet_name='unrolled')

In [ ]:
key_cols = ['subreg_id', 'year']
value_cols = ['total_hhpop', 'total_hh', 'total_emp']

print('Shape check')
print('  current :', df_current.shape)
print('  original:', df_original.shape)
print('  row count match:', len(df_current) == len(df_original))
print()

print('Column check')
print('  current columns :', list(df_current.columns))
print('  original columns:', list(df_original.columns))
print('  columns match:', list(df_current.columns) == list(df_original.columns))
print()

print('Key integrity check')
print('  current duplicate keys :', df_current.duplicated(key_cols).sum())
print('  original duplicate keys:', df_original.duplicated(key_cols).sum())
print()

merged = df_current.merge(
    df_original,
    on=key_cols,
    how='outer',
    suffixes=('_current', '_original'),
    indicator=True,
)

print('Coverage check')
print(merged['_merge'].value_counts(dropna=False).rename_axis('status').to_frame('rows'))
print()

matched = merged.loc[merged['_merge'] == 'both'].copy()
for col in value_cols:
    matched[f'{col}_diff'] = matched[f'{col}_current'] - matched[f'{col}_original']

summary = pd.DataFrame(
    {
        'exact_match_rows': [
            (matched[f'{col}_current'] == matched[f'{col}_original']).sum() for col in value_cols
        ],
        'mismatch_rows': [
            (matched[f'{col}_current'] != matched[f'{col}_original']).sum() for col in value_cols
        ],
        'max_abs_diff': [matched[f'{col}_diff'].abs().max() for col in value_cols],
        'sum_current': [matched[f'{col}_current'].sum() for col in value_cols],
        'sum_original': [matched[f'{col}_original'].sum() for col in value_cols],
        'sum_diff': [matched[f'{col}_diff'].sum() for col in value_cols],
    },
    index=value_cols,
)

print('Value comparison summary')
display(summary)

mismatch_mask = False
for col in value_cols:
    mismatch_mask = mismatch_mask | (matched[f'{col}_current'] != matched[f'{col}_original'])

mismatches = matched.loc[
    mismatch_mask,
    key_cols
    + [f'{col}_current' for col in value_cols]
    + [f'{col}_original' for col in value_cols]
    + [f'{col}_diff' for col in value_cols],
].sort_values(key_cols)

print('Sample mismatched rows')
display(mismatches.head(20))
print('Total mismatched rows:', len(mismatches))
print()

current_year_totals = df_current.groupby('year')[value_cols].sum().sort_index()
original_year_totals = df_original.groupby('year')[value_cols].sum().sort_index()
year_diff = (
    current_year_totals
    .rename(columns=lambda col: f'{col}_current')
    .join(original_year_totals.rename(columns=lambda col: f'{col}_original'))
)
for col in value_cols:
    year_diff[f'{col}_diff'] = year_diff[f'{col}_current'] - year_diff[f'{col}_original']

print('Year-level totals comparison')
display(year_diff)

In [ ]:
mismatches.query('total_hh_diff != 0')[['subreg_id', 'year', 'total_hh_current', 'total_hh_original', 'total_hh_diff']].head(20)